# 🔬 Phase 3: Early-Stopping Ablation Study & Extended Evaluation
**Paper:** Early-Stopping Activation Steering for Hallucination Mitigation in Vietnamese Domain-Specific RAG

**Core Question Addressed:** Does early-stopping provide genuine advantages over full (continuous) steering?

**Key Innovations Tested:**
1. **Cosine Decay Early-Stopping** — smooth α annealing eliminates activation shock
2. **Fine-grained α tuning** — dedicated α optimized for finite K (not reusing K=∞ optimal)
3. **Extended generation** (max_new_tokens=200) — reveals over-steering degradation in full steering
4. **Multi-metric evaluation** — ROUGE-L + BERTScore + 4-gram Repetition Rate + EOS Rate

---
**Environment:** Kaggle GPU T4 ×2, Internet ON  
**Input Dataset:** `steering-phase1-artifacts` (Phase 1 output)  
**Estimated Runtime:** ~10–11 hours

In [ ]:
# Cell 1: Environment Setup
!pip install -q bitsandbytes accelerate transformers torch rouge-score bert-score tqdm
print('✅ Dependencies installed!')

In [ ]:
# Cell 2: Imports & Global Configuration
import os, json, glob, random, time, math, gc
import numpy as np
import torch
from tqdm import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

OUTPUT_DIR = '/kaggle/working'

print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_mem / 1e9:.1f} GB)')

In [ ]:
# Cell 3: Data Loading & Reproducible Split (IDENTICAL to Phase 1/2)
DATA_FILENAME = 'vietnamese_medical_halueval_15k_specialized.json'
search_paths = [
    f'/kaggle/input/**/{DATA_FILENAME}',
    f'/kaggle/input/{DATA_FILENAME}',
    f'data/{DATA_FILENAME}',
    f'./{DATA_FILENAME}'
]

data_path = None
for pattern in search_paths:
    matches = glob.glob(pattern, recursive=True)
    if matches:
        data_path = matches[0]
        break

if not data_path:
    raise FileNotFoundError(f'❌ Could not locate {DATA_FILENAME}')

with open(data_path, 'r', encoding='utf-8') as f:
    raw_dataset = json.load(f)

# CRITICAL: Use identical shuffle + split as Phase 1/2 for reproducibility
shuffled_records = list(raw_dataset)
random.seed(SEED)  # Reset seed to ensure identical shuffle
random.shuffle(shuffled_records)

n_total = len(shuffled_records)
n_train = int(n_total * 0.70)
n_val = int(n_total * 0.15)

train_records = shuffled_records[:n_train]
val_records = shuffled_records[n_train:n_train + n_val]
test_records = shuffled_records[n_train + n_val:]

print(f'📊 Total: {n_total:,} | Train: {len(train_records):,} | Val: {len(val_records):,} | Test: {len(test_records):,}')

In [ ]:
# Cell 4: Load Phase 1 Artifacts
config_paths = glob.glob('/kaggle/input/**/steering_config.json', recursive=True)
v_steer_paths = glob.glob('/kaggle/input/**/v_steer.pt', recursive=True)
v_rand_paths = glob.glob('/kaggle/input/**/v_rand.pt', recursive=True)

if not config_paths:
    raise FileNotFoundError('❌ steering_config.json not found. Upload Phase 1 output as Kaggle Input Dataset.')

with open(config_paths[0], 'r', encoding='utf-8') as f:
    steering_config = json.load(f)

BEST_LAYER = steering_config['best_layer']
PHASE1_ALPHA = steering_config['best_alpha']
PHASE1_K = steering_config['best_K']

v_steer = torch.load(v_steer_paths[0], map_location='cpu')
v_rand = torch.load(v_rand_paths[0], map_location='cpu')

print(f'✅ Phase 1 Artifacts Loaded:')
print(f'   Layer: {BEST_LAYER} | Alpha: {PHASE1_ALPHA} | K: {PHASE1_K}')
print(f'   v_steer: {v_steer.shape} (norm={v_steer.float().norm():.4f})')
print(f'   v_rand:  {v_rand.shape} (norm={v_rand.float().norm():.4f})')

In [ ]:
# Cell 5: Load Model (Qwen2.5-7B-Instruct 4-bit NF4)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'

print(f'⌛ Loading {MODEL_NAME} in 4-bit NF4...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True
)
model.eval()
print('✅ Model loaded!')

In [ ]:
# Cell 6: Steering Hook with Multiple Decay Strategies
# =====================================================
# This is the CORE INNOVATION of Phase 3.
#
# Phase 2 used HARD CUTOFF: full alpha for t < K, then 0.
# This causes 'activation shock' — a sudden Δ=alpha jump in
# the residual stream, destabilizing generation.
#
# Phase 3 introduces SMOOTH DECAY strategies:
#   - Cosine: α(t) = α₀ · ½(1 + cos(πt/K))  [smooth, recommended]
#   - Linear: α(t) = α₀ · (1 − t/K)          [simple baseline]
#
# Both decay from α₀ → 0 over K steps with NO discontinuity.
# =====================================================

PROMPT_TEMPLATE = """Dựa vào ngữ cảnh y học sau đây, hãy trả lời câu hỏi:
Ngữ cảnh: {context}
Câu hỏi: {question}
Trả lời: """

class SteeringHook:
    """
    Unified activation steering hook with configurable decay.
    
    Decay types:
      'hard'   : Full alpha for step < K, then 0  (Phase 2 baseline)
      'cosine' : Smooth cosine annealing α₀ → 0 over K steps
      'linear' : Linear decay α₀ → 0 over K steps
    
    For K=999 or K=inf, steering runs at full alpha for all steps.
    """
    
    def __init__(self, layer_idx, v_vector, alpha=20.0, K=8, decay='hard'):
        self.layer_idx = layer_idx
        self.v_vector = v_vector
        self.alpha = alpha
        self.K = K
        self.decay = decay
        self.step_counter = 0
        self.handle = None
    
    def _effective_alpha(self, t):
        """Compute effective steering intensity at generation step t."""
        # K=999 means 'steer all steps' (legacy Phase 1/2 convention)
        if self.K >= 999:
            return self.alpha
        if t >= self.K:
            return 0.0
        if self.decay == 'hard':
            return self.alpha
        elif self.decay == 'cosine':
            return self.alpha * 0.5 * (1.0 + math.cos(math.pi * t / self.K))
        elif self.decay == 'linear':
            return self.alpha * (1.0 - t / self.K)
        return self.alpha
    
    def hook_fn(self, module, inputs, output):
        eff_a = self._effective_alpha(self.step_counter)
        if eff_a > 0:
            if isinstance(output, tuple):
                h = output[0]
                v = self.v_vector.to(h.device).to(h.dtype)
                h[:, -1, :] = h[:, -1, :] + eff_a * v
                output = (h,) + output[1:]
            else:
                v = self.v_vector.to(output.device).to(output.dtype)
                output[:, -1, :] = output[:, -1, :] + eff_a * v
        self.step_counter += 1
        return output
    
    def register(self, model):
        layer_module = model.model.layers[self.layer_idx]
        self.step_counter = 0
        self.handle = layer_module.register_forward_hook(self.hook_fn)
    
    def remove(self):
        if self.handle:
            self.handle.remove()
            self.handle = None
    
    def __repr__(self):
        K_str = 'inf' if self.K >= 999 else str(self.K)
        return f'SteeringHook(a={self.alpha}, K={K_str}, decay={self.decay})'


def compute_repetition_ratio(text, n=4):
    """4-gram repetition ratio. Higher = more repetitive text."""
    words = text.split()
    if len(words) < n:
        return 0.0
    ngrams = [tuple(words[i:i+n]) for i in range(len(words) - n + 1)]
    if not ngrams:
        return 0.0
    return 1.0 - len(set(ngrams)) / len(ngrams)


print('✅ SteeringHook with Cosine/Linear Decay defined.')
# Quick verification of decay schedule
h = SteeringHook(0, v_steer, alpha=20.0, K=8, decay='cosine')
schedule = [h._effective_alpha(t) for t in range(12)]
print(f'Cosine decay schedule (α=20, K=8): {[f"{a:.1f}" for a in schedule]}')

In [ ]:
# Cell 7: Core Evaluation Engine
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def evaluate_condition(model, tokenizer, records, v_vector, alpha, K, decay,
                       max_new_tokens=80, condition_name='', layer_idx=None):
    """
    Run generation + ROUGE-L evaluation for one experimental condition.
    Returns list of per-sample dicts with generated text, scores, metadata.
    """
    results = []
    total_start = time.time()
    
    for idx, rec in enumerate(tqdm(records, desc=f'{condition_name}')):
        ctx = rec.get('knowledge_context', rec.get('context', ''))
        q = rec['question']
        ref = rec['right_answer']
        prompt = PROMPT_TEMPLATE.format(context=ctx, question=q)
        inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
        
        # Register hook if steering is active
        hook = None
        if v_vector is not None and layer_idx is not None:
            hook = SteeringHook(layer_idx, v_vector, alpha=alpha, K=K, decay=decay)
            hook.register(model)
        
        # Generate with per-sample seed for paired comparison
        torch.manual_seed(SEED + idx)
        t0 = time.time()
        with torch.no_grad():
            out_ids = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.1,
                top_p=0.85
            )
        elapsed_ms = (time.time() - t0) * 1000
        
        if hook:
            hook.remove()
        
        # Decode and score
        gen_tokens = out_ids[0][inputs.input_ids.shape[1]:]
        gen_text = tokenizer.decode(gen_tokens, skip_special_tokens=True)
        r_score = scorer.score(ref, gen_text)['rougeL'].fmeasure * 100
        
        # Check EOS vs truncation
        eos_id = tokenizer.eos_token_id
        hit_eos = bool(len(gen_tokens) > 0 and gen_tokens[-1].item() == eos_id)
        
        results.append({
            'idx': idx,
            'question': q,
            'reference': ref,
            'generated': gen_text,
            'category': rec.get('hallucination_type', 'unknown'),
            'rouge_l': r_score,
            'num_tokens': len(gen_tokens),
            'elapsed_ms': elapsed_ms,
            'hit_eos': hit_eos,
            'rep_4gram': compute_repetition_ratio(gen_text, n=4),
        })
    
    # Summary statistics
    avg_rl = np.mean([r['rouge_l'] for r in results])
    avg_len = np.mean([r['num_tokens'] for r in results])
    avg_lat = np.mean([r['elapsed_ms'] for r in results])
    avg_rep = np.mean([r['rep_4gram'] for r in results])
    eos_pct = np.mean([r['hit_eos'] for r in results]) * 100
    tput = avg_len / (avg_lat / 1000) if avg_lat > 0 else 0
    total_min = (time.time() - total_start) / 60
    
    print(f'\n  ✅ [{condition_name}] Done in {total_min:.1f} min')
    print(f'     ROUGE-L: {avg_rl:.2f}% | Len: {avg_len:.1f} | Rep: {avg_rep:.4f} | EOS: {eos_pct:.1f}% | {tput:.1f} tok/s')
    
    return results

print('✅ Evaluation engine ready.')

---
## Part A: Validation Sweep — Finding Optimal Cosine Decay Config

**Goal:** Evaluate 10 `(α, K, decay)` configurations on 50 validation samples to identify the best early-stopping setup.

**Rationale:** Phase 1 found `α=20, K=999` as optimal, but K=999 ≡ K=∞ (no early stopping).
The grid sweep showed that at α=15, K=16 **beats** K=∞ on validation.
Cosine decay should amplify this advantage by eliminating activation shock.

**Estimated time:** ~80 minutes

In [ ]:
# Cell 8: Part A — Validation Sweep (50 samples × 10 configs)
print('='*70)
print('PART A: VALIDATION SWEEP — Cosine Decay Early-Stopping')
print('='*70)

val_subset = val_records[:50]

# Validation baseline
print('\n--- Vanilla Baseline on Validation ---')
val_baseline = evaluate_condition(
    model, tokenizer, val_subset,
    v_vector=None, alpha=0, K=0, decay='hard',
    max_new_tokens=80, condition_name='Val Baseline'
)
val_bl_rouge = np.mean([r['rouge_l'] for r in val_baseline])

# Configurations to sweep
# Key insight: α=20 is optimal for K=∞ but too aggressive for finite K.
# We test moderate α values (8-18) with K=8 and K=16.
val_configs = [
    # Cosine decay — primary innovation
    {'alpha': 8.0,  'K': 8,  'decay': 'cosine'},
    {'alpha': 12.0, 'K': 8,  'decay': 'cosine'},
    {'alpha': 15.0, 'K': 8,  'decay': 'cosine'},
    {'alpha': 18.0, 'K': 8,  'decay': 'cosine'},
    {'alpha': 8.0,  'K': 16, 'decay': 'cosine'},
    {'alpha': 12.0, 'K': 16, 'decay': 'cosine'},
    {'alpha': 15.0, 'K': 16, 'decay': 'cosine'},
    {'alpha': 18.0, 'K': 16, 'decay': 'cosine'},
    # Linear decay — alternative
    {'alpha': 15.0, 'K': 16, 'decay': 'linear'},
    {'alpha': 18.0, 'K': 16, 'decay': 'linear'},
]

val_sweep = {}
for cfg in val_configs:
    name = f"{cfg['decay']}_a{cfg['alpha']}_K{cfg['K']}"
    results = evaluate_condition(
        model, tokenizer, val_subset,
        v_vector=v_steer, alpha=cfg['alpha'], K=cfg['K'], decay=cfg['decay'],
        max_new_tokens=80, condition_name=name, layer_idx=BEST_LAYER
    )
    avg_rl = np.mean([r['rouge_l'] for r in results])
    val_sweep[name] = {'config': cfg, 'rouge_l': avg_rl, 'results': results}

# Also test Full Steering on validation for fair comparison
print('\n--- Full Steering (α=20, K=∞) on Validation ---')
val_full = evaluate_condition(
    model, tokenizer, val_subset,
    v_vector=v_steer, alpha=20.0, K=999, decay='hard',
    max_new_tokens=80, condition_name='Val Full Steering', layer_idx=BEST_LAYER
)
val_full_rouge = np.mean([r['rouge_l'] for r in val_full])

# Summary table
print('\n' + '='*70)
print('VALIDATION SWEEP RESULTS')
print('='*70)
print(f'{"Config":<30} {"ROUGE-L":>10} {"vs Baseline":>12} {"vs Full":>10}')
print('-'*65)
print(f'{"Vanilla Baseline":<30} {val_bl_rouge:>9.2f}% {"---":>12} {"---":>10}')
print(f'{"Full Steering (a=20,K=inf)":<30} {val_full_rouge:>9.2f}% {"+" + f"{val_full_rouge-val_bl_rouge:.2f}":>12} {"---":>10}')

sorted_sweep = sorted(val_sweep.items(), key=lambda x: x[1]['rouge_l'], reverse=True)
for name, info in sorted_sweep:
    rl = info['rouge_l']
    d_bl = rl - val_bl_rouge
    d_full = rl - val_full_rouge
    s_bl = f'+{d_bl:.2f}' if d_bl >= 0 else f'{d_bl:.2f}'
    s_full = f'+{d_full:.2f}' if d_full >= 0 else f'{d_full:.2f}'
    marker = ' ⭐' if d_bl > 0 and d_full > 0 else (' ✓' if d_bl > 0 else '')
    print(f'{name:<30} {rl:>9.2f}% {s_bl:>12} {s_full:>10}{marker}')

# Select best configs
best1_name = sorted_sweep[0][0]
best1_cfg = sorted_sweep[0][1]['config']
best2_name = sorted_sweep[1][0] if len(sorted_sweep) > 1 else best1_name
best2_cfg = sorted_sweep[1][1]['config'] if len(sorted_sweep) > 1 else best1_cfg

print(f'\n🏆 Best Early-Stop: {best1_name}')
print(f'🥈 Runner-up:      {best2_name}')

# Save validation checkpoint
val_checkpoint = {
    'baseline_rouge_l': val_bl_rouge,
    'full_steering_rouge_l': val_full_rouge,
    'sweep': {n: {'config': i['config'], 'rouge_l': i['rouge_l']} for n, i in val_sweep.items()},
    'best_config': best1_cfg,
    'runner_up_config': best2_cfg,
}
with open(os.path.join(OUTPUT_DIR, 'phase3_val_sweep.json'), 'w') as f:
    json.dump(val_checkpoint, f, indent=2)
print('💾 Validation sweep saved.')

gc.collect()
torch.cuda.empty_cache()

---
## Part B: Test Evaluation (max_new_tokens=80)

**6 experimental conditions × 500 test samples:**
1. Vanilla Baseline (no steering)
2. 🆕 Best Cosine Decay Early-Stop (from Part A)
3. 🆕 Runner-up Cosine Decay Early-Stop (from Part A)
4. Full Steering (α=20, K=∞) — Phase 2 replication
5. Control: Random Direction
6. Control: Sign-Flipped

**Estimated time:** ~8.5 hours

In [ ]:
# Cell 9: Part B — Test Set Evaluation (max_new_tokens=80)
print('='*70)
print('PART B: TEST SET EVALUATION (max_new_tokens=80)')
print('='*70)

TEST_LIMIT = min(500, len(test_records))
test_subset = test_records[:TEST_LIMIT]

all_test_results = {}

# 1. Vanilla Baseline
print(f'\n--- [1/6] Vanilla Baseline ({TEST_LIMIT} samples) ---')
all_test_results['baseline'] = evaluate_condition(
    model, tokenizer, test_subset,
    v_vector=None, alpha=0, K=0, decay='hard',
    max_new_tokens=80, condition_name='Baseline'
)

# 2. Best Cosine Decay Early-Stop
print(f'\n--- [2/6] Best ES: {best1_name} ---')
all_test_results['es_best'] = evaluate_condition(
    model, tokenizer, test_subset,
    v_vector=v_steer, alpha=best1_cfg['alpha'], K=best1_cfg['K'], decay=best1_cfg['decay'],
    max_new_tokens=80, condition_name=f'ES Best ({best1_name})', layer_idx=BEST_LAYER
)

# 3. Runner-up Cosine Decay Early-Stop
print(f'\n--- [3/6] Runner-up ES: {best2_name} ---')
all_test_results['es_runnerup'] = evaluate_condition(
    model, tokenizer, test_subset,
    v_vector=v_steer, alpha=best2_cfg['alpha'], K=best2_cfg['K'], decay=best2_cfg['decay'],
    max_new_tokens=80, condition_name=f'ES 2nd ({best2_name})', layer_idx=BEST_LAYER
)

# 4. Full Steering (replicate Phase 2)
print(f'\n--- [4/6] Full Steering (α=20, K=∞) ---')
all_test_results['full_steering'] = evaluate_condition(
    model, tokenizer, test_subset,
    v_vector=v_steer, alpha=20.0, K=999, decay='hard',
    max_new_tokens=80, condition_name='Full Steering', layer_idx=BEST_LAYER
)

# 5. Control: Random Direction
print(f'\n--- [5/6] Control: Random Direction ---')
all_test_results['ctrl_random'] = evaluate_condition(
    model, tokenizer, test_subset,
    v_vector=v_rand, alpha=20.0, K=999, decay='hard',
    max_new_tokens=80, condition_name='Ctrl: Random', layer_idx=BEST_LAYER
)

# 6. Control: Sign-Flipped
print(f'\n--- [6/6] Control: Sign-Flipped ---')
all_test_results['ctrl_signflip'] = evaluate_condition(
    model, tokenizer, test_subset,
    v_vector=-v_steer, alpha=20.0, K=999, decay='hard',
    max_new_tokens=80, condition_name='Ctrl: Sign-Flip', layer_idx=BEST_LAYER
)

# Save checkpoint (exclude generated text to keep JSON small)
checkpoint_80 = {}
for cond, results in all_test_results.items():
    checkpoint_80[cond] = [
        {k: v for k, v in r.items() if k not in ('generated', 'reference', 'question')}
        for r in results
    ]
with open(os.path.join(OUTPUT_DIR, 'phase3_test80_checkpoint.json'), 'w') as f:
    json.dump(checkpoint_80, f, indent=2)
print('\n💾 Test (80 tok) checkpoint saved.')

gc.collect()
torch.cuda.empty_cache()

---
## Part C: Extended Generation (max_new_tokens=200)

**Hypothesis:** Full steering at α=20 over 200 tokens causes **over-steering degradation** 
(repetitive text, style collapse), while early-stopping maintains natural generation quality.

This is the most critical experiment for proving early-stopping's practical value.

**3 conditions × 100 test samples**  
**Estimated time:** ~1.5 hours

In [ ]:
# Cell 10: Part C — Extended Generation Test (max_new_tokens=200)
print('='*70)
print('PART C: EXTENDED GENERATION TEST (max_new_tokens=200)')
print('='*70)

EXT_LIMIT = 100  # Subset for time efficiency
ext_subset = test_subset[:EXT_LIMIT]

ext_results = {}

# 1. Baseline (200 tok)
print(f'\n--- [1/3] Baseline (200 tok, {EXT_LIMIT} samples) ---')
ext_results['baseline_200'] = evaluate_condition(
    model, tokenizer, ext_subset,
    v_vector=None, alpha=0, K=0, decay='hard',
    max_new_tokens=200, condition_name='Baseline (200)'
)

# 2. Best Early-Stop (200 tok)
print(f'\n--- [2/3] Best ES (200 tok) ---')
ext_results['es_best_200'] = evaluate_condition(
    model, tokenizer, ext_subset,
    v_vector=v_steer, alpha=best1_cfg['alpha'], K=best1_cfg['K'], decay=best1_cfg['decay'],
    max_new_tokens=200, condition_name=f'ES Best (200)', layer_idx=BEST_LAYER
)

# 3. Full Steering (200 tok)
print(f'\n--- [3/3] Full Steering (200 tok) ---')
ext_results['full_200'] = evaluate_condition(
    model, tokenizer, ext_subset,
    v_vector=v_steer, alpha=20.0, K=999, decay='hard',
    max_new_tokens=200, condition_name='Full Steering (200)', layer_idx=BEST_LAYER
)

# Save checkpoint
checkpoint_200 = {}
for cond, results in ext_results.items():
    checkpoint_200[cond] = [
        {k: v for k, v in r.items() if k not in ('generated', 'reference', 'question')}
        for r in results
    ]
with open(os.path.join(OUTPUT_DIR, 'phase3_test200_checkpoint.json'), 'w') as f:
    json.dump(checkpoint_200, f, indent=2)
print('\n💾 Extended test (200 tok) checkpoint saved.')

gc.collect()
torch.cuda.empty_cache()

---
## Part D: BERTScore Multi-Metric Evaluation

Compute BERTScore F1 (semantic similarity) using `bert-base-multilingual-cased`.
This metric captures meaning preservation better than ROUGE-L's surface n-gram overlap.

**Estimated time:** ~10-15 minutes

In [ ]:
# Cell 11: Part D — BERTScore Computation
print('='*70)
print('PART D: BERTSCORE COMPUTATION')
print('='*70)

try:
    from bert_score import score as bert_score_fn
    
    # Compute BERTScore for all test (80 tok) conditions
    for cond_name, results in all_test_results.items():
        refs = [r['reference'] for r in results]
        hyps = [r['generated'] for r in results]
        print(f'  Computing BERTScore for {cond_name} ({len(refs)} samples)...')
        P, R, F1 = bert_score_fn(hyps, refs,
                                  model_type='bert-base-multilingual-cased',
                                  num_layers=9, verbose=False, device='cuda')
        f1_list = F1.tolist()
        for r, bs in zip(results, f1_list):
            r['bertscore_f1'] = bs
        print(f'    → Avg BERTScore F1: {np.mean(f1_list):.4f}')
    
    # Compute BERTScore for extended (200 tok) conditions
    for cond_name, results in ext_results.items():
        refs = [r['reference'] for r in results]
        hyps = [r['generated'] for r in results]
        print(f'  Computing BERTScore for {cond_name} ({len(refs)} samples)...')
        P, R, F1 = bert_score_fn(hyps, refs,
                                  model_type='bert-base-multilingual-cased',
                                  num_layers=9, verbose=False, device='cuda')
        f1_list = F1.tolist()
        for r, bs in zip(results, f1_list):
            r['bertscore_f1'] = bs
        print(f'    → Avg BERTScore F1: {np.mean(f1_list):.4f}')
    
    print('\n✅ BERTScore computed for all conditions.')
    
except Exception as e:
    print(f'⚠️ BERTScore computation failed: {e}')
    print('Continuing without BERTScore...')

---
## Part E: Category-Specific Breakdown & Statistical Analysis

In [ ]:
# Cell 12: Part E — Category Breakdown + Paired Statistical Test
print('='*70)
print('PART E: CATEGORY BREAKDOWN & STATISTICAL ANALYSIS')
print('='*70)

# --- Category Breakdown ---
categories = set()
for r in all_test_results.get('baseline', []):
    categories.add(r['category'])
categories = sorted(categories)

print(f'\nCategories found: {categories}')
print(f'\n{"Category":<35} {"Baseline":>10} {"ES Best":>10} {"Full Steer":>10} {"Δ(ES-BL)":>10} {"Δ(ES-Full)":>10}')
print('-'*90)

for cat in categories:
    bl_scores = [r['rouge_l'] for r in all_test_results['baseline'] if r['category'] == cat]
    es_scores = [r['rouge_l'] for r in all_test_results['es_best'] if r['category'] == cat]
    fs_scores = [r['rouge_l'] for r in all_test_results['full_steering'] if r['category'] == cat]
    if bl_scores and es_scores and fs_scores:
        bl_avg = np.mean(bl_scores)
        es_avg = np.mean(es_scores)
        fs_avg = np.mean(fs_scores)
        print(f'{cat:<35} {bl_avg:>9.2f}% {es_avg:>9.2f}% {fs_avg:>9.2f}% {es_avg-bl_avg:>+9.2f} {es_avg-fs_avg:>+9.2f}')

# --- Paired Bootstrap Test (ES Best vs Baseline) ---
print('\n--- Paired Bootstrap Significance Test ---')
bl_scores = np.array([r['rouge_l'] for r in all_test_results['baseline']])
es_scores = np.array([r['rouge_l'] for r in all_test_results['es_best']])
fs_scores = np.array([r['rouge_l'] for r in all_test_results['full_steering']])

n_bootstrap = 10000
n_samples = len(bl_scores)

def bootstrap_p_value(scores_a, scores_b, n_boot=10000):
    """One-sided test: H1 = mean(A) > mean(B)"""
    observed_diff = np.mean(scores_a) - np.mean(scores_b)
    count = 0
    for _ in range(n_boot):
        idx = np.random.randint(0, len(scores_a), size=len(scores_a))
        boot_diff = np.mean(scores_a[idx]) - np.mean(scores_b[idx])
        if boot_diff <= 0:
            count += 1
    return count / n_boot

p_es_vs_bl = bootstrap_p_value(es_scores, bl_scores)
p_es_vs_fs = bootstrap_p_value(es_scores, fs_scores)
p_fs_vs_bl = bootstrap_p_value(fs_scores, bl_scores)

print(f'  ES Best vs Baseline:       Δ = {np.mean(es_scores)-np.mean(bl_scores):+.2f} pp, p = {p_es_vs_bl:.4f}')
print(f'  ES Best vs Full Steering:  Δ = {np.mean(es_scores)-np.mean(fs_scores):+.2f} pp, p = {p_es_vs_fs:.4f}')
print(f'  Full Steer vs Baseline:    Δ = {np.mean(fs_scores)-np.mean(bl_scores):+.2f} pp, p = {p_fs_vs_bl:.4f}')

In [ ]:
# Cell 13: Qualitative Examples — Show generated text samples
print('='*70)
print('QUALITATIVE EXAMPLES (5 random test samples)')
print('='*70)

np.random.seed(42)
sample_indices = np.random.choice(len(all_test_results['baseline']), size=5, replace=False)

for i, idx in enumerate(sample_indices):
    bl = all_test_results['baseline'][idx]
    es = all_test_results['es_best'][idx]
    fs = all_test_results['full_steering'][idx]
    
    print(f'\n--- Example {i+1} (idx={idx}, category={bl["category"]}) ---')
    print(f'Question:  {bl["question"][:100]}...')
    print(f'Reference: {bl["reference"][:100]}...')
    print(f'\nBaseline     (ROUGE-L={bl["rouge_l"]:.1f}%): {bl["generated"][:150]}...')
    print(f'ES Best      (ROUGE-L={es["rouge_l"]:.1f}%): {es["generated"][:150]}...')
    print(f'Full Steer   (ROUGE-L={fs["rouge_l"]:.1f}%): {fs["generated"][:150]}...')

In [ ]:
# Cell 14: FINAL COMPREHENSIVE RESULTS TABLE
print('\n' + '='*100)
print('📊 PHASE 3: FINAL COMPREHENSIVE RESULTS')
print('='*100)

def summarize(results):
    return {
        'rouge_l': np.mean([r['rouge_l'] for r in results]),
        'bertscore': np.mean([r.get('bertscore_f1', 0) for r in results]),
        'rep_4gram': np.mean([r['rep_4gram'] for r in results]),
        'eos_rate': np.mean([r['hit_eos'] for r in results]) * 100,
        'avg_len': np.mean([r['num_tokens'] for r in results]),
        'latency': np.mean([r['elapsed_ms'] for r in results]),
    }

# --- Table 1: Test Results (max_new_tokens=80) ---
print(f'\n📋 Table 1: Test Results (max_new_tokens=80, N={TEST_LIMIT})')
print(f'{"Method":<32} {"ROUGE-L%":>9} {"BERTScore":>10} {"Rep4gram":>9} {"EOS%":>6} {"Len":>5} {"ms":>7}')
print('-'*85)

display_order = ['baseline', 'es_best', 'es_runnerup', 'full_steering', 'ctrl_random', 'ctrl_signflip']
display_names = {
    'baseline': 'Vanilla Baseline',
    'es_best': f'ES Cosine Best ({best1_name})',
    'es_runnerup': f'ES 2nd ({best2_name})',
    'full_steering': 'Full Steering (α=20,K=∞)',
    'ctrl_random': 'Ctrl: Random Direction',
    'ctrl_signflip': 'Ctrl: Sign-Flipped (-v)',
}

for key in display_order:
    if key in all_test_results:
        s = summarize(all_test_results[key])
        name = display_names.get(key, key)[:32]
        print(f'{name:<32} {s["rouge_l"]:>8.2f}% {s["bertscore"]:>9.4f} {s["rep_4gram"]:>8.4f} {s["eos_rate"]:>5.1f} {s["avg_len"]:>5.1f} {s["latency"]:>6.0f}')

# --- Table 2: Extended Results (max_new_tokens=200) ---
print(f'\n📋 Table 2: Extended Generation (max_new_tokens=200, N={EXT_LIMIT})')
print(f'{"Method":<32} {"ROUGE-L%":>9} {"BERTScore":>10} {"Rep4gram":>9} {"EOS%":>6} {"Len":>5} {"ms":>7}')
print('-'*85)

ext_display = {
    'baseline_200': 'Vanilla Baseline (200)',
    'es_best_200': f'ES Cosine Best (200)',
    'full_200': 'Full Steering (200)',
}

for key in ['baseline_200', 'es_best_200', 'full_200']:
    if key in ext_results:
        s = summarize(ext_results[key])
        name = ext_display.get(key, key)[:32]
        print(f'{name:<32} {s["rouge_l"]:>8.2f}% {s["bertscore"]:>9.4f} {s["rep_4gram"]:>8.4f} {s["eos_rate"]:>5.1f} {s["avg_len"]:>5.1f} {s["latency"]:>6.0f}')

print('\n' + '='*100)

In [ ]:
# Cell 15: Export ALL Results to JSON
print('='*70)
print('EXPORTING FINAL RESULTS')
print('='*70)

# Full export with all generated text (for paper supplementary)
full_export = {
    'experiment': 'Phase 3: Early-Stopping Ablation Study',
    'model': 'Qwen/Qwen2.5-7B-Instruct (4-bit NF4)',
    'best_early_stop_config': best1_cfg,
    'runner_up_config': best2_cfg,
    'phase1_config': {'layer': BEST_LAYER, 'alpha': PHASE1_ALPHA, 'K': PHASE1_K},
    'test_80_summary': {k: summarize(v) for k, v in all_test_results.items()},
    'test_200_summary': {k: summarize(v) for k, v in ext_results.items()},
    'validation_sweep': val_checkpoint,
}

with open(os.path.join(OUTPUT_DIR, 'phase3_full_results.json'), 'w', encoding='utf-8') as f:
    json.dump(full_export, f, indent=2, ensure_ascii=False)

# Save generated texts for qualitative analysis
texts_export = {}
for cond, results in {**all_test_results, **ext_results}.items():
    texts_export[cond] = [
        {'idx': r['idx'], 'generated': r['generated'], 'reference': r['reference'],
         'rouge_l': r['rouge_l'], 'category': r['category']}
        for r in results
    ]

with open(os.path.join(OUTPUT_DIR, 'phase3_generated_texts.json'), 'w', encoding='utf-8') as f:
    json.dump(texts_export, f, indent=2, ensure_ascii=False)

print(f'✅ Exported:')
print(f'   - phase3_full_results.json (summary metrics)')
print(f'   - phase3_generated_texts.json (all generated text)')
print(f'   - phase3_val_sweep.json (validation sweep)')
print(f'   - phase3_test80_checkpoint.json')
print(f'   - phase3_test200_checkpoint.json')
print('\n🎉 PHASE 3 COMPLETE!')